# 👁️ Retinal Disease Detection Using Deep Learning on OCT Images
### 🔬 End-to-End Diagnostic Pipeline: NumPy Fundamentals, Deep CNNs, Transfer Learning, Explainable AI (Grad-CAM), and Model Serialization (.pkl)

---

## 📋 Executive Overview & Clinical Context
Optical Coherence Tomography (OCT) is a non-invasive, micron-resolution imaging modality widely used in clinical ophthalmology to capture cross-sectional anatomical cross-sections of the human retina. This comprehensive Jupyter Notebook provides an end-to-end deep learning framework for the automated classification and explainable diagnosis of major retinal pathologies:

1. **CNV (Choroidal Neovascularization):** Abnormal proliferation of fragile blood vessels under the retinal pigment epithelium (RPE), common in exudative ('wet') Age-Related Macular Degeneration (AMD).
2. **DME (Diabetic Macular Edema):** Microvascular leakage causing fluid accumulation and intraretinal cystoid spaces in diabetic patients.
3. **DRUSEN:** Acellular lipid and proteinaceous deposits between the RPE and Bruch's membrane, the clinical hallmark of dry AMD.
4. **NORMAL:** Clear retinal lamellar architecture without subretinal or intraretinal fluid, preserved foveal depression.

---

### 🚀 Notebook Roadmap:
1. **Environment & Compute Setup** (PyTorch, Torchvision, CUDA detection)
2. **Neural Network Fundamentals from First Principles** (Pure NumPy MLP, ReLU vs. Sigmoid vs. Tanh)
3. **Dataset Ingestion & Preprocessing** (Kermany2017 OCT dataset, Stratified split, Data augmentations)
4. **Deep Learning Architectures** (Custom 4-stage CNN from scratch + Transfer Learning: ResNet18, MobileNetV2, DenseNet121, VGG16)
5. **Training Routine with Regularization** (AdamW, L2 weight decay, ReduceLROnPlateau, Early Stopping)
6. **Multi-Metric Evaluation & Benchmark Comparison** (Accuracy, Precision, Recall, Macro F1, Confusion Matrices)
7. **Explainable AI with Grad-CAM** (Biomarker localization & anatomical heatmaps)
8. **Production Model Serialization** (Packaging models, metrics, and state dicts into `.pkl` format)
9. **Single-Scan Clinical Inference** (Interactive test on novel patient scans)

## 1. Environment & Compute Hardware Setup
We configure plotting parameters, random seeds, and verify compute acceleration (NVIDIA CUDA / CPU).

In [ ]:
import os
import io
import sys
import copy
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Plotting configuration
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style="whitegrid")

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Hardware detection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"Compute Device : {device}")
if device.type == "cuda":
    print(f"GPU Name       : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("=" * 60)

## 2. Neural Network Fundamentals from First Principles (Pure NumPy)
Before training deep convolutional networks, we implement the mathematical mechanics of neural networks from scratch:
- **Activation Functions:** Sigmoid, ReLU, Tanh, and Softmax.
- **Analytical Derivatives:** Backward gradient flow for each activation.
- **Multi-Class Cross-Entropy Loss & Softmax Gradient.**
- **2-Layer Multi-Layer Perceptron (MLP):** Full matrix forward propagation, analytical backpropagation, L2 weight decay, and momentum SGD.
- **Activation Function Benchmark Experiment:** Comparing convergence speed and final accuracy across ReLU, Sigmoid, and Tanh.

In [ ]:
class Activation:
    @staticmethod
    def sigmoid(x):
        clipped = np.clip(x, -500, 500)
        return 1.0 / (1.0 + np.exp(-clipped))

    @staticmethod
    def sigmoid_derivative(x):
        s = Activation.sigmoid(x)
        return s * (1.0 - s)

    @staticmethod
    def relu(x):
        return np.maximum(0, x)

    @staticmethod
    def relu_derivative(x):
        return (x > 0).astype(float)

    @staticmethod
    def tanh(x):
        return np.tanh(x)

    @staticmethod
    def tanh_derivative(x):
        return 1.0 - np.tanh(x) ** 2

    @staticmethod
    def softmax(x):
        shift_x = x - np.max(x, axis=1, keepdims=True)
        exps = np.exp(shift_x)
        return exps / np.sum(exps, axis=1, keepdims=True)


class Loss:
    @staticmethod
    def cross_entropy(y_pred, y_true):
        eps = 1e-12
        n_samples = y_pred.shape[0]
        p = np.clip(y_pred[np.arange(n_samples), y_true], eps, 1.0 - eps)
        return -np.mean(np.log(p))

    @staticmethod
    def cross_entropy_grad(y_pred, y_true):
        n_samples = y_pred.shape[0]
        grad = y_pred.copy()
        grad[np.arange(n_samples), y_true] -= 1.0
        return grad / n_samples


class NumPyMLP:
    """2-Layer Multi-Layer Perceptron built purely from scratch using NumPy."""
    def __init__(self, input_dim, hidden_dim, output_dim, activation="relu", seed=42):
        np.random.seed(seed)
        self.activation_name = activation.lower()
        if self.activation_name == "relu":
            scale1 = np.sqrt(2.0 / input_dim)
            scale2 = np.sqrt(2.0 / hidden_dim)
        else:
            scale1 = np.sqrt(1.0 / input_dim)
            scale2 = np.sqrt(1.0 / hidden_dim)

        self.W1 = np.random.randn(input_dim, hidden_dim) * scale1
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * scale2
        self.b2 = np.zeros((1, output_dim))

        self.v_W1 = np.zeros_like(self.W1)
        self.v_b1 = np.zeros_like(self.b1)
        self.v_W2 = np.zeros_like(self.W2)
        self.v_b2 = np.zeros_like(self.b2)

    def _activate(self, z):
        if self.activation_name == "relu": return Activation.relu(z)
        elif self.activation_name == "sigmoid": return Activation.sigmoid(z)
        elif self.activation_name == "tanh": return Activation.tanh(z)
        raise ValueError(f"Unknown activation: {self.activation_name}")

    def _activate_derivative(self, z):
        if self.activation_name == "relu": return Activation.relu_derivative(z)
        elif self.activation_name == "sigmoid": return Activation.sigmoid_derivative(z)
        elif self.activation_name == "tanh": return Activation.tanh_derivative(z)
        raise ValueError(f"Unknown activation: {self.activation_name}")

    def forward(self, X):
        self.X = X
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self._activate(self.Z1)
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.probs = Activation.softmax(self.Z2)
        return self.probs

    def backward(self, y_true):
        dZ2 = Loss.cross_entropy_grad(self.probs, y_true)
        self.dW2 = np.dot(self.A1.T, dZ2)
        self.db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self._activate_derivative(self.Z1)
        self.dW1 = np.dot(self.X.T, dZ1)
        self.db1 = np.sum(dZ1, axis=0, keepdims=True)

    def step(self, lr=0.01, momentum=0.9, l2_reg=1e-4):
        self.dW2 += l2_reg * self.W2
        self.dW1 += l2_reg * self.W1

        self.v_W2 = momentum * self.v_W2 - lr * self.dW2
        self.v_b2 = momentum * self.v_b2 - lr * self.db2
        self.v_W1 = momentum * self.v_W1 - lr * self.dW1
        self.v_b1 = momentum * self.v_b1 - lr * self.db1

        self.W2 += self.v_W2
        self.b2 += self.v_b2
        self.W1 += self.v_W1
        self.b1 += self.v_b1

    def train_epoch(self, X, y, lr=0.01, momentum=0.9, l2_reg=1e-4):
        probs = self.forward(X)
        loss = Loss.cross_entropy(probs, y)
        self.backward(y)
        self.step(lr=lr, momentum=momentum, l2_reg=l2_reg)
        preds = np.argmax(probs, axis=1)
        acc = np.mean(preds == y)
        return loss, acc

In [ ]:
# Run Activation Function Comparison Experiment
np.random.seed(42)
n_samples, n_features, n_classes = 400, 32, 4
X_syn = np.random.randn(n_samples, n_features)
y_syn = (np.sum(X_syn[:, :4] ** 2, axis=1) + np.sin(X_syn[:, 4]) * 2).astype(int) % n_classes

activation_results = {}
epochs = 60
for act in ["relu", "sigmoid", "tanh"]:
    mlp = NumPyMLP(input_dim=n_features, hidden_dim=64, output_dim=n_classes, activation=act)
    hist = []
    for ep in range(epochs):
        loss_val, acc_val = mlp.train_epoch(X_syn, y_syn, lr=0.05, momentum=0.9)
        hist.append((loss_val, acc_val))
    activation_results[act] = hist

# Plot convergence comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
color_map = {"relu": "#2ca02c", "sigmoid": "#1f77b4", "tanh": "#ff7f0e"}

for act, hist in activation_results.items():
    losses = [h[0] for h in hist]
    accs = [h[1] * 100 for h in hist]
    ax1.plot(range(1, epochs + 1), losses, label=act.upper(), color=color_map[act], linewidth=2.2)
    ax2.plot(range(1, epochs + 1), accs, label=act.upper(), color=color_map[act], linewidth=2.2)

ax1.set_title("Training Loss Convergence by Activation", fontsize=12, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross Entropy Loss")
ax1.legend()
ax1.grid(True, linestyle=":", alpha=0.6)

ax2.set_title("Training Accuracy Convergence by Activation", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.legend()
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

print("NumPy MLP Convergence Benchmark Summary:")
for act, hist in activation_results.items():
    print(f"  Activation: {act.upper():<8} -> Final Loss: {hist[-1][0]:.4f} | Final Accuracy: {hist[-1][1]*100:.1f}%")

## 3. Dataset Ingestion, Preprocessing & Stratified Splitting
We load the Kermany2017 OCT dataset.
- Target classes: `['CNV', 'DME', 'DRUSEN', 'NORMAL']`.
- Data Augmentations: Random horizontal flips, random rotations (±10°), affine scaling, and color jitter.
- Partitioning: 70% Train, 15% Validation, 15% Test with stratified class balance.
- Standard ImageNet normalization: `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`.

In [ ]:
CLASS_NAMES = ["CNV", "DME", "DRUSEN", "NORMAL"]
NUM_CLASSES = len(CLASS_NAMES)

CLASS_DESCRIPTIONS = {
    "CNV": "Choroidal Neovascularization: Pathologic growth of abnormal blood vessels beneath the retina, causing severe vision loss and subretinal fluid accumulation.",
    "DME": "Diabetic Macular Edema: Accumulation of fluid in the macula due to leaking blood vessels from diabetic retinopathy, leading to retinal thickening.",
    "DRUSEN": "Drusen: Yellow extracellular lipid deposits beneath the retinal pigment epithelium (RPE), characteristic early sign of Age-related Macular Degeneration (AMD).",
    "NORMAL": "Normal Retina: Preserved retinal layer architecture with continuous foveal contour and no intraretinal or subretinal fluid/deposits."
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_transforms(img_size=224):
    train_transform = transforms.Compose([
        transforms.Lambda(lambda img: img.convert("RGB") if hasattr(img, "convert") else img),
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.RandomAffine(degrees=0, scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    eval_transform = transforms.Compose([
        transforms.Lambda(lambda img: img.convert("RGB") if hasattr(img, "convert") else img),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    return train_transform, eval_transform


class OCTDataset(Dataset):
    """PyTorch Dataset for Retinal OCT Images supporting PIL, bytes, and paths."""
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        item = self.images[idx]
        if isinstance(item, bytes):
            img = Image.open(io.BytesIO(item)).convert("RGB")
        elif isinstance(item, Image.Image):
            img = item.convert("RGB")
        elif isinstance(item, str):
            img = Image.open(item).convert("RGB")
        else:
            raise TypeError(f"Unsupported image type: {type(item)}")

        if self.transform is not None:
            img = self.transform(img)

        label = int(self.labels[idx])
        return img, label


def load_kermany_oct_data(cache_dir=None, val_ratio=0.15, test_ratio=0.15, random_state=42):
    parquet_path = hf_hub_download(
        repo_id="zacharielegault/Kermany2017-OCT",
        filename="data/test-00000-of-00001.parquet",
        repo_type="dataset",
        cache_dir=cache_dir
    )

    table = pq.read_table(parquet_path)
    images_bytes = [table["image"][i]["bytes"].as_py() for i in range(len(table))]
    labels = [table["label"][i].as_py() for i in range(len(table))]

    labels_np = np.array(labels)
    total_samples = len(labels)

    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=test_ratio, random_state=random_state)
    train_val_idx, test_idx = next(sss_test.split(range(total_samples), labels_np))

    val_rel_ratio = val_ratio / (1.0 - test_ratio)
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=val_rel_ratio, random_state=random_state)
    train_sub_idx, val_sub_idx = next(sss_val.split(train_val_idx, labels_np[train_val_idx]))

    train_idx = train_val_idx[train_sub_idx]
    val_idx = train_val_idx[val_sub_idx]

    data_splits = {
        "train_images": [images_bytes[i] for i in train_idx],
        "train_labels": [labels[i] for i in train_idx],
        "val_images": [images_bytes[i] for i in val_idx],
        "val_labels": [labels[i] for i in val_idx],
        "test_images": [images_bytes[i] for i in test_idx],
        "test_labels": [labels[i] for i in test_idx],
    }

    print(f"Dataset successfully loaded: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")
    return data_splits


def get_oct_dataloaders(data_splits, batch_size=32, img_size=224, num_workers=0):
    train_tf, eval_tf = get_transforms(img_size=img_size)
    train_ds = OCTDataset(data_splits["train_images"], data_splits["train_labels"], transform=train_tf)
    val_ds = OCTDataset(data_splits["val_images"], data_splits["val_labels"], transform=eval_tf)
    test_ds = OCTDataset(data_splits["test_images"], data_splits["test_labels"], transform=eval_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return train_loader, val_loader, test_loader

In [ ]:
# Load dataset and create dataloaders
data_splits = load_kermany_oct_data()
train_loader, val_loader, test_loader = get_oct_dataloaders(data_splits, batch_size=32)

# Visualize Class Distribution Across Splits
records = []
for split_name, labels in [("Train", data_splits["train_labels"]), ("Val", data_splits["val_labels"]), ("Test", data_splits["test_labels"])]:
    for lbl in labels:
        records.append({"Split": split_name, "Class": CLASS_NAMES[lbl]})
df_dist = pd.DataFrame(records)

plt.figure(figsize=(9, 5))
sns.countplot(data=df_dist, x="Class", hue="Split", palette="mako")
plt.title("Retinal OCT Dataset Split Class Distribution", fontsize=13, fontweight="bold")
plt.xlabel("OCT Retinal Condition", fontsize=11)
plt.ylabel("Number of Scans", fontsize=11)
plt.grid(axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Display representative sample OCT scans from each of the 4 retinal classes
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
samples_by_class = {}
for img_b, lbl in zip(data_splits["test_images"], data_splits["test_labels"]):
    if lbl not in samples_by_class:
        samples_by_class[lbl] = img_b
    if len(samples_by_class) == 4:
        break

for i, (lbl, img_b) in enumerate(sorted(samples_by_class.items())):
    img = Image.open(io.BytesIO(img_b)).convert("RGB")
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(f"Class {lbl}: {CLASS_NAMES[lbl]}", fontsize=12, fontweight="bold")
    axes[i].axis("off")

plt.suptitle("Clinical OCT B-Scan Categories (Kermany2017)", fontsize=14, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

## 4. Deep Learning Model Architectures
We define and benchmark 5 model architectures:
1. **CustomOCTCNN**: Custom 4-stage convolutional architecture built from scratch.
2. **ResNet18**: Residual Network with skip connections.
3. **MobileNetV2**: Efficient inverted residual blocks for edge deployment.
4. **DenseNet121**: Densely connected convolutional network.
5. **VGG16**: Classic deep convolutional network.

In [ ]:
class CustomOCTCNN(nn.Module):
    """Custom Convolutional Neural Network built from scratch using PyTorch."""
    def __init__(self, num_classes=4, dropout_rate=0.4):
        super(CustomOCTCNN, self).__init__()
        
        self.features = nn.Sequential(
            # Stage 1: 3 -> 32 channels, 224x224 -> 112x112
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Stage 2: 32 -> 64 channels, 112x112 -> 56x56
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Stage 3: 64 -> 128 channels, 56x56 -> 28x28
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Stage 4: 128 -> 256 channels, 28x28 -> 14x14
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=False),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=False),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 64),
            nn.ReLU(inplace=False),
            nn.Dropout(dropout_rate * 0.75),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        feat = self.features(x)
        out = self.classifier(feat)
        return out


def build_resnet18(num_classes=4, pretrained=True, freeze_backbone=False):
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model


def build_vgg16(num_classes=4, pretrained=True, freeze_backbone=False):
    weights = models.VGG16_Weights.DEFAULT if pretrained else None
    model = models.vgg16(weights=weights)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model


def build_mobilenet_v2(num_classes=4, pretrained=True, freeze_backbone=False):
    weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
    model = models.mobilenet_v2(weights=weights)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model


def build_densenet121(num_classes=4, pretrained=True, freeze_backbone=False):
    weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
    model = models.densenet121(weights=weights)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model


SUPPORTED_MODELS = {
    "custom_cnn": CustomOCTCNN,
    "resnet18": build_resnet18,
    "mobilenet_v2": build_mobilenet_v2,
    "densenet121": build_densenet121,
    "vgg16": build_vgg16,
}


def build_model(model_name="resnet18", num_classes=4, pretrained=True, freeze_backbone=False):
    name = model_name.lower()
    if name == "custom_cnn":
        return CustomOCTCNN(num_classes=num_classes)
    elif name in SUPPORTED_MODELS:
        builder = SUPPORTED_MODELS[name]
        return builder(num_classes=num_classes, pretrained=pretrained, freeze_backbone=freeze_backbone)
    else:
        raise ValueError(f"Unknown model name: {model_name}. Supported: {list(SUPPORTED_MODELS.keys())}")


# Count parameters for each architecture
param_counts = {}
for m_name in SUPPORTED_MODELS.keys():
    m = build_model(m_name, num_classes=4, pretrained=False)
    total_params = sum(p.numel() for p in m.parameters())
    trainable_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    param_counts[m_name.upper()] = {"Total Parameters": total_params, "Trainable Parameters": trainable_params}

df_params = pd.DataFrame(param_counts).T
print("Model Parameter Counts:")
print(df_params)

## 5. Training Pipeline: Regularization, Scheduling & Optimization
- **Optimizer:** `AdamW` with L2 weight decay ($1 	imes 10^{-4}$).
- **Scheduler:** `ReduceLROnPlateau` reducing learning rate when validation loss plateaus.
- **Early Stopping:** Monitors validation loss with patience to prevent overfitting.

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()
        total += labels.size(0)

    return running_loss / total, (correct / total) * 100.0


def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data).item()
            total += labels.size(0)

    return running_loss / total, (correct / total) * 100.0


def train_model(
    model, train_loader, val_loader, num_epochs=10, lr=1e-3, weight_decay=1e-4, patience=3, device=None, model_name="Model"
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    best_val_loss = float("inf")
    best_weights = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    print(f"\n--- Training {model_name} on {device} (Epochs: {num_epochs}, LR: {lr}) ---")
    start_time = time.time()

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)
        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        print(f"Epoch {epoch+1:02d}/{num_epochs:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | LR: {current_lr:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered at epoch {epoch+1} (No improvement for {patience} epochs).")
                break

    print(f"Training completed in {time.time() - start_time:.2f}s. Restoring best weights (Val Loss: {best_val_loss:.4f}).")
    model.load_state_dict(best_weights)
    return model, history

## 6. Evaluation Metrics Engine
Calculates Accuracy, Precision (Macro & Weighted), Recall (Macro & Weighted), F1-Score (Macro & Weighted), Confusion Matrix, and class-wise breakdowns.

In [ ]:
def evaluate_model(model, test_loader, device=None, class_names=None):
    if class_names is None:
        class_names = CLASS_NAMES
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()
    model.to(device)

    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            all_targets.extend(targets.numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    all_probs = np.array(all_probs)

    acc = float(accuracy_score(all_targets, all_preds))
    prec_macro = float(precision_score(all_targets, all_preds, average="macro", zero_division=0))
    rec_macro = float(recall_score(all_targets, all_preds, average="macro", zero_division=0))
    f1_macro = float(f1_score(all_targets, all_preds, average="macro", zero_division=0))

    prec_weighted = float(precision_score(all_targets, all_preds, average="weighted", zero_division=0))
    rec_weighted = float(recall_score(all_targets, all_preds, average="weighted", zero_division=0))
    f1_weighted = float(f1_score(all_targets, all_preds, average="weighted", zero_division=0))

    cm = confusion_matrix(all_targets, all_preds)
    report_dict = classification_report(all_targets, all_preds, target_names=class_names, output_dict=True, zero_division=0)
    report_str = classification_report(all_targets, all_preds, target_names=class_names, zero_division=0)

    class_metrics = {}
    for i, c_name in enumerate(class_names):
        class_metrics[c_name] = {
            "precision": float(report_dict[c_name]["precision"]),
            "recall": float(report_dict[c_name]["recall"]),
            "f1-score": float(report_dict[c_name]["f1-score"]),
            "support": int(report_dict[c_name]["support"])
        }

    misclassified_indices = np.where(all_preds != all_targets)[0]
    misclassifications = []
    for idx in misclassified_indices:
        misclassifications.append({
            "index": int(idx),
            "true_label": class_names[all_targets[idx]],
            "predicted_label": class_names[all_preds[idx]],
            "confidence": float(all_probs[idx][all_preds[idx]])
        })

    return {
        "accuracy": acc,
        "precision_macro": prec_macro,
        "recall_macro": rec_macro,
        "f1_macro": f1_macro,
        "precision_weighted": prec_weighted,
        "recall_weighted": rec_weighted,
        "f1_weighted": f1_weighted,
        "confusion_matrix": cm,
        "classification_report": report_str,
        "class_metrics": class_metrics,
        "num_test_samples": len(all_targets),
        "num_misclassified": len(misclassified_indices),
        "misclassifications": misclassifications
    }

## 7. Model Execution & Evaluation on Test Dataset
We load cached models or train on the fly, then evaluate each on unseen test data.

In [ ]:
models_to_run = [
    ("vgg16", {"epochs": 3, "lr": 1e-3, "l2": 1e-4, "freeze": True}),
    ("resnet18", {"epochs": 4, "lr": 1e-3, "l2": 1e-4, "freeze": True}),
    ("densenet121", {"epochs": 4, "lr": 1e-3, "l2": 1e-4, "freeze": True}),
    ("mobilenet_v2", {"epochs": 4, "lr": 1e-3, "l2": 1e-4, "freeze": True}),
    ("custom_cnn", {"epochs": 6, "lr": 1e-3, "l2": 1e-4, "freeze": False}),
]

trained_models = {}
eval_metrics = {}
confusion_matrices = {}
benchmark_records = []

os.makedirs("models", exist_ok=True)

for model_name, cfg in models_to_run:
    pkl_path = f"models/{model_name}.pkl"
    pt_path = f"models/{model_name}.pt"
    
    loaded = False
    if os.path.exists(pkl_path):
        try:
            with open(pkl_path, "rb") as f:
                pkg = pickle.load(f)
            model = pkg["model"]
            print(f"Loaded cached {model_name.upper()} from {pkl_path}")
            loaded = True
        except Exception as e:
            print(f"Failed loading {pkl_path}: {e}")
            
    if not loaded and os.path.exists(pt_path):
        try:
            model = build_model(model_name, num_classes=NUM_CLASSES, pretrained=False)
            model.load_state_dict(torch.load(pt_path, map_location=device))
            print(f"Loaded weights for {model_name.upper()} from {pt_path}")
            loaded = True
        except Exception as e:
            print(f"Failed loading {pt_path}: {e}")

    if not loaded:
        print(f"Training {model_name.upper()} from scratch...")
        model = build_model(model_name, num_classes=NUM_CLASSES, pretrained=True, freeze_backbone=cfg["freeze"])
        model, hist = train_model(model, train_loader, val_loader, num_epochs=cfg["epochs"], lr=cfg["lr"], weight_decay=cfg["l2"], device=device, model_name=model_name.upper())

    # Evaluate on test set
    metrics = evaluate_model(model, test_loader, device=device, class_names=CLASS_NAMES)
    trained_models[model_name] = model
    eval_metrics[model_name] = metrics
    confusion_matrices[model_name] = metrics["confusion_matrix"]
    
    benchmark_records.append({
        "Model": model_name.upper(),
        "Accuracy": metrics["accuracy"],
        "Precision": metrics["precision_macro"],
        "Recall": metrics["recall_macro"],
        "F1-Score": metrics["f1_macro"],
        "Misclassified": metrics["num_misclassified"]
    })
    print(f"  --> {model_name.upper()} | Accuracy: {metrics['accuracy']*100:.2f}% | F1: {metrics['f1_macro']:.4f}")

benchmark_df = pd.DataFrame(benchmark_records).sort_values(by="F1-Score", ascending=False)
print("\n" + "=" * 65)
print("BENCHMARK COMPARISON TABLE (TEST SPLIT):")
print("=" * 65)
print(benchmark_df.to_string(index=False))
print("=" * 65)

## 8. Benchmark Visualizations: Comparison Charts & Confusion Matrices

In [ ]:
# Model Comparison Grouped Bar Chart
fig, ax = plt.subplots(figsize=(11, 6))
model_list = benchmark_df["Model"].tolist()
metrics_keys = ["Accuracy", "Precision", "Recall", "F1-Score"]
x = np.arange(len(model_list))
width = 0.2
colors = ["#1f77b4", "#2ca02c", "#ff7f0e", "#9467bd"]

for i, metric in enumerate(metrics_keys):
    values = [benchmark_df.loc[benchmark_df["Model"] == m, metric].values[0] for m in model_list]
    rects = ax.bar(x + (i - 1.5) * width, values, width, label=metric, color=colors[i], alpha=0.9)
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f"{h:.2f}", xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8)

ax.set_ylabel("Score (0.0 - 1.0)", fontsize=11, fontweight="bold")
ax.set_title("Comprehensive Model Benchmark Comparison (Test Set)", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(model_list, fontsize=10, fontweight="bold")
ax.set_ylim(0, 1.15)
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle=":", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix Grid
cols = 3
rows = (len(confusion_matrices) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5.5 * cols, 5 * rows))
axes = axes.flatten()

for idx, (m_name, cm) in enumerate(confusion_matrices.items()):
    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"{m_name.upper()} (Test Set)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted Condition", fontsize=10)
    ax.set_ylabel("True Condition", fontsize=10)

for j in range(len(confusion_matrices), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## 9. Explainable AI with Grad-CAM (Biomarker Localization)
We visualize where the best model looks to make its predictions using Gradient-weighted Class Activation Mapping (Grad-CAM).

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer=None):
        self.model = model
        self.model.eval()
        for m in self.model.modules():
            if hasattr(m, "inplace"):
                m.inplace = False
        self.target_layer = target_layer or self._find_default_target_layer()
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _find_default_target_layer(self):
        if hasattr(self.model, "layer4"):
            return self.model.layer4[-1]
        elif hasattr(self.model, "features"):
            for module in reversed(list(self.model.features.modules())):
                if isinstance(module, (nn.Conv2d, nn.BatchNorm2d)):
                    return module
            return self.model.features[-1]
        else:
            last_conv = None
            for module in self.model.modules():
                if isinstance(module, nn.Conv2d):
                    last_conv = module
            if last_conv is not None:
                return last_conv
            raise ValueError("Could not automatically locate a convolutional layer for Grad-CAM.")

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach().clone()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach().clone()

        self.f_hook = self.target_layer.register_forward_hook(forward_hook)
        self.b_hook = self.target_layer.register_full_backward_hook(backward_hook)

    def remove_hooks(self):
        if hasattr(self, "f_hook") and self.f_hook is not None:
            self.f_hook.remove()
        if hasattr(self, "b_hook") and self.b_hook is not None:
            self.b_hook.remove()

    def generate_heatmap(self, input_tensor, target_class=None):
        for param in self.model.parameters():
            param.requires_grad_(True)
        self.model.zero_grad()
        output = self.model(input_tensor)
        probs = F.softmax(output, dim=1).detach().cpu().numpy()[0]
        
        pred_class = int(np.argmax(probs))
        confidence = float(probs[pred_class])
        if target_class is None:
            target_class = pred_class

        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1.0
        output.backward(gradient=one_hot, retain_graph=True)

        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(input_tensor.shape[2], input_tensor.shape[3]), mode="bilinear", align_corners=False)
        cam_np = cam.squeeze().cpu().numpy()

        cam_min, cam_max = cam_np.min(), cam_np.max()
        if cam_max - cam_min > 1e-8:
            heatmap = (cam_np - cam_min) / (cam_max - cam_min)
        else:
            heatmap = np.zeros_like(cam_np)

        return heatmap, pred_class, confidence, probs

    @staticmethod
    def overlay_heatmap_on_image(original_image, heatmap, alpha=0.45, colormap="jet"):
        if isinstance(original_image, Image.Image):
            pil_img = original_image.convert("RGB")
            w, h = pil_img.size
        else:
            pil_img = Image.fromarray(original_image).convert("RGB")
            w, h = pil_img.size

        import matplotlib as mpl
        try:
            cmap = mpl.colormaps[colormap]
        except AttributeError:
            import matplotlib.cm as cm
            cmap = cm.get_cmap(colormap)
        colored_heatmap = cmap(heatmap)[:, :, :3]
        colored_heatmap_uint8 = (colored_heatmap * 255).astype(np.uint8)
        heatmap_img = Image.fromarray(colored_heatmap_uint8).resize((w, h), Image.Resampling.BILINEAR)

        blended = Image.blend(pil_img, heatmap_img, alpha=alpha)
        return blended, heatmap_img

In [ ]:
# Generate Grad-CAM Explainability Gallery for Top Model (VGG16 / Best Model)
best_model_name = benchmark_df.iloc[0]["Model"].lower()
best_model = trained_models[best_model_name]
print(f"Generating Grad-CAM visualizations using Best Model: {best_model_name.upper()}")

_, eval_tf = get_transforms()
gradcam = GradCAM(best_model)

target_samples = {}
for i, lbl in enumerate(data_splits["test_labels"]):
    if lbl not in target_samples:
        target_samples[lbl] = data_splits["test_images"][i]
    if len(target_samples) == 4:
        break

fig, axes = plt.subplots(4, 3, figsize=(13, 14))
for row, (lbl, img_bytes) in enumerate(sorted(target_samples.items())):
    pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    tensor_img = eval_tf(pil_img).unsqueeze(0).to(device)

    heatmap, pred_c, conf, probs = gradcam.generate_heatmap(tensor_img)
    blended, heat_pil = GradCAM.overlay_heatmap_on_image(pil_img, heatmap, alpha=0.45)

    c_name = CLASS_NAMES[lbl]
    p_name = CLASS_NAMES[pred_c]

    # Original
    axes[row, 0].imshow(pil_img, cmap="gray")
    axes[row, 0].set_title(f"True: {c_name} (Original OCT)", fontsize=11, fontweight="bold")
    axes[row, 0].axis("off")

    # Heatmap
    axes[row, 1].imshow(heatmap, cmap="jet")
    axes[row, 1].set_title("Grad-CAM Activation Map", fontsize=11, fontweight="bold")
    axes[row, 1].axis("off")

    # Overlay
    axes[row, 2].imshow(blended)
    axes[row, 2].set_title(f"Pred: {p_name} ({conf*100:.1f}%) Overlay", fontsize=11, fontweight="bold")
    axes[row, 2].axis("off")

gradcam.remove_hooks()
plt.suptitle(f"Model Explainability: Grad-CAM Retinal Layer Heatmaps ({best_model_name.upper()})", fontsize=14, fontweight="bold", y=0.995)
plt.tight_layout()
plt.show()

## 10. Production Model Serialization (.pkl) & Artifact Export
Serializes the trained model object, PyTorch weights, evaluation metrics, and metadata to `.pkl` format.

In [ ]:
def save_model_package_pkl(model, model_name, metrics, filepath, extra_info=None):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    for m in model.modules():
        if hasattr(m, "_forward_hooks"): m._forward_hooks.clear()
        if hasattr(m, "_backward_hooks"): m._backward_hooks.clear()
        if hasattr(m, "_forward_pre_hooks"): m._forward_pre_hooks.clear()
        if hasattr(m, "_full_backward_hooks"): m._full_backward_hooks.clear()

    package = {
        "model_name": model_name,
        "model_architecture": str(model.__class__.__name__),
        "state_dict": model.state_dict(),
        "model": model,
        "metrics": metrics,
        "class_names": CLASS_NAMES,
        "input_size": (3, 224, 224),
        "extra_info": extra_info or {}
    }
    with open(filepath, "wb") as f:
        pickle.dump(package, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Successfully serialized model package to PKL: {filepath}")

# Save Best Model Package
best_row = benchmark_df.iloc[0]
best_model_name = best_row["Model"].lower()
best_model = trained_models[best_model_name]

save_model_package_pkl(
    model=best_model,
    model_name=best_model_name,
    metrics=best_row.to_dict(),
    filepath="models/best_retinal_oct_model.pkl",
    extra_info={"benchmark_table": benchmark_df.to_dict(orient="records")}
)

# Also save PyTorch state_dict
torch.save(best_model.state_dict(), "models/best_retinal_oct_model.pt")

# Verify loading from PKL
with open("models/best_retinal_oct_model.pkl", "rb") as f:
    loaded_pkg = pickle.load(f)
print("\nPKL Package Verification:")
print(f"  Model Name: {loaded_pkg['model_name']}")
print(f"  Accuracy  : {loaded_pkg['metrics']['Accuracy']*100:.2f}%")
print(f"  Macro F1  : {loaded_pkg['metrics']['F1-Score']:.4f}")
print(f"  Classes   : {loaded_pkg['class_names']}")

## 11. Interactive Clinical Diagnostic Inference on Single OCT Scan

In [ ]:
def diagnose_oct_scan(image_input, model=None, top_model_name="vgg16"):
    """Performs end-to-end diagnostic inference and Grad-CAM explainability on an OCT image."""
    if model is None:
        if top_model_name in trained_models:
            model = trained_models[top_model_name]
        else:
            with open("models/best_retinal_oct_model.pkl", "rb") as f:
                model = pickle.load(f)["model"]

    if isinstance(image_input, str):
        pil_img = Image.open(image_input).convert("RGB")
    elif isinstance(image_input, bytes):
        pil_img = Image.open(io.BytesIO(image_input)).convert("RGB")
    else:
        pil_img = image_input.convert("RGB")

    _, eval_tf = get_transforms()
    tensor_img = eval_tf(pil_img).unsqueeze(0).to(device)

    gradcam = GradCAM(model)
    heatmap, pred_c, conf, probs = gradcam.generate_heatmap(tensor_img)
    blended, _ = GradCAM.overlay_heatmap_on_image(pil_img, heatmap, alpha=0.45)
    gradcam.remove_hooks()

    pred_class_name = CLASS_NAMES[pred_c]

    # Visualization
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.5))

    ax1.imshow(pil_img, cmap="gray")
    ax1.set_title("Input OCT B-Scan", fontsize=11, fontweight="bold")
    ax1.axis("off")

    ax2.imshow(blended)
    ax2.set_title(f"Grad-CAM: {pred_class_name} ({conf*100:.1f}%)", fontsize=11, fontweight="bold")
    ax2.axis("off")

    colors = ["#d9534f" if i == pred_c else "#428bca" for i in range(len(CLASS_NAMES))]
    y_pos = np.arange(len(CLASS_NAMES))
    ax3.barh(y_pos, probs * 100, color=colors)
    ax3.set_yticks(y_pos)
    ax3.set_yticklabels(CLASS_NAMES, fontsize=10, fontweight="bold")
    ax3.set_xlabel("Predicted Probability (%)", fontsize=10)
    ax3.set_title("Class Probability Distribution", fontsize=11, fontweight="bold")
    ax3.set_xlim(0, 100)
    for i, v in enumerate(probs * 100):
        ax3.text(v + 1.5, i, f"{v:.1f}%", va="center", fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.show()

    print(f"Diagnostic Result: {pred_class_name} ({conf*100:.2f}% Confidence)")
    print(f"Pathology Description: {CLASS_DESCRIPTIONS[pred_class_name]}")


# Test inference on sample image
sample_test_path = "assets/sample_images/DME_sample_1.jpeg"
if os.path.exists(sample_test_path):
    print(f"Running Diagnostic Inference on sample: {sample_test_path}")
    diagnose_oct_scan(sample_test_path, model=best_model)
else:
    print("Testing on first sample in test split:")
    diagnose_oct_scan(data_splits["test_images"][0], model=best_model)

## 12. Conclusion & Summary of Deliverables
- **NumPy Fundamentals Baseline:** Demonstrates mathematical grasp of forward/backprop and explains why ReLU avoids saturation compared to Sigmoid and Tanh.
- **Computer Vision Preprocessing:** Handles automated downloading, image byte decoding, stratified data splits, and robust data augmentations.
- **Deep CNN Benchmarking:** Compares custom CNN against 4 transfer learning baselines (ResNet18, MobileNetV2, DenseNet121, VGG16).
- **Explainable AI (Grad-CAM):** Pinpoints anatomical retinal biomarkers for clinical interpretability.
- **Model Serialization (.pkl):** Full self-contained models exported to `.pkl` standard for downstream deployment in web applications and clinical prototypes.